# LOSSIA: where should maintenance act?

**Scope:** a small, complete analysis of line L5, with intuitive charts and no interface.
Identify operations needing attention, distinguish own stops from external waiting,
and show whether problems recur across the four weeks. Preserve the source files.

**Decision rules:** prioritize loss of availability from own stops; inspect slow cycles
separately; use frequent short stops as a microstop investigation signal. Low induced
stoppage is a clue from the challenge, not proof of causality. Daily totals cannot
identify individual faults or establish which machine stopped another one.

The configuration includes parallel operations. Its order supports visualization,
but does not establish a complete dependency graph or unique upstream/downstream links.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 24)
plt.rcParams.update({'figure.figsize': (11, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})
COLORS = {'TBF': '#2a9d8f', 'TAP': '#e76f51', 'DES': '#457b9d', 'SAT': '#b39ddb'}

# Accumulated times are treated as hours, following the team's 22-hour briefing.
EXPECTED_HOURS = 22
TIME_TOLERANCE_HOURS = 0.10  # screening tolerance; not a cleaning rule
AVAILABILITY_TARGET = 90.0
CYCLE_TARGET_CMIN = 80.0  # target from the presentation; 80 cmin = 48 seconds
STOP_RATE_BASE = 100  # latest team notes; slides use 1000: confirm before delivery
RATE_LABEL = f'Own stops / {STOP_RATE_BASE} pieces'
# Editable screening heuristic, not a definition supplied by the client.
MICROSTOP_MEAN_SECONDS = 60

ROOT = Path.cwd()
if not (ROOT / 'data' / 'datos_hackaton.csv').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
data = pd.read_csv(DATA_DIR / 'datos_hackaton.csv')
config = pd.read_csv(DATA_DIR / 'config_hackaton.csv')
print(f'Data: {len(data):,} rows; configuration: {len(config)} operations')
display(data.head())
display(config.head())

Data: 1,820 rows; configuration: 91 operations


,Linea,Year,Week,Jour,Operacion,TBF,TAP,DES,SAT,NAP,NPR
0,VLL_MONTAJE_L5,2026,40,1,MANOP749,0.0,22.0,0.0,0.0,0,0
1,VLL_MONTAJE_L5,2026,40,1,Ma_BM1_2,0.0,22.0,0.0,0.0,0,0
2,VLL_MONTAJE_L5,2026,40,1,Ma_BM2_2,0.0,22.0,0.0,0.0,0,0
3,VLL_MONTAJE_L5,2026,40,1,Ma_VESRE,0.0,22.0,0.0,0.0,0,0
4,VLL_MONTAJE_L5,2026,40,1,Ma_VES_0,0.0,22.0,0.0,0.0,0,0


,Linea,Operacion,Orden_OP
0,VLL_MONTAJE_L5,ST_0015,1
1,VLL_MONTAJE_L5,ST_0020,2
2,VLL_MONTAJE_L5,ST_0030,3
3,VLL_MONTAJE_L5,ST_0040,4
4,VLL_MONTAJE_L5,ST-0050,5


## 1. Validate before drawing conclusions

The actual grain is **line + operation + year + week + day** (`Jour`), rather than
one row per week. Check keys, coverage, types, nonnegative values and configuration
matches. Keep operation names unchanged: hyphens and underscores are meaningful identifiers.
The configuration is supplied clean; these checks verify that the join is safe.

In [3]:
JOIN_KEYS = ['Linea', 'Operacion']
DAY_KEYS = ['Linea', 'Year', 'Week', 'Jour', 'Operacion']
TIMES = ['TBF', 'TAP', 'DES', 'SAT']
COUNTERS = ['NAP', 'NPR']
VALUES = TIMES + COUNTERS
required = set(DAY_KEYS + VALUES)
assert required.issubset(data.columns), f'Missing data columns: {required - set(data.columns)}'
assert set(JOIN_KEYS + ['Orden_OP']).issubset(config.columns), 'Incomplete configuration'

numeric_columns = ['Year', 'Week', 'Jour'] + VALUES
numeric = data[numeric_columns].apply(pd.to_numeric, errors='coerce')
quality = pd.Series({
    'Missing values (all columns)': int(data.isna().sum().sum()),
    'Non-numeric values in numeric columns': int((numeric.isna() & data[numeric_columns].notna()).sum().sum()),
    'Non-finite numeric values': int((numeric.notna() & ~np.isfinite(numeric)).sum().sum()),
    'Negative time/counter values': int(numeric[VALUES].lt(0).sum().sum()),
    'Duplicate daily keys': int(data.duplicated(DAY_KEYS).sum()),
    'Duplicate configuration keys': int(config.duplicated(JOIN_KEYS).sum()),
    'Missing configuration keys/order': int(config[JOIN_KEYS + ['Orden_OP']].isna().sum().sum()),
    'Non-integer counters': int((numeric[COUNTERS].notna() & numeric[COUNTERS].mod(1).ne(0)).sum().sum()),
}, name='Count')
display(quality.to_frame())
assert quality.eq(0).all(), 'Resolve the reported structural issues before calculating indicators.'
data = data.copy()
data[numeric_columns] = numeric
joined = data.merge(config, on=JOIN_KEYS, how='left', validate='many_to_one', indicator=True)
unmatched = joined.loc[joined['_merge'].ne('both'), JOIN_KEYS].drop_duplicates()
unused_config = config.merge(data[JOIN_KEYS].drop_duplicates(), on=JOIN_KEYS, how='left', indicator=True)
unused_config = unused_config.loc[unused_config['_merge'].eq('left_only'), JOIN_KEYS]
display(unmatched.rename_axis('Unmatched data operations'))
display(unused_config.rename_axis('Configured operations with no observations'))
assert unmatched.empty, 'Some data operations have no configuration; review identifiers.'
joined = joined.drop(columns='_merge')
coverage = joined.groupby(['Year', 'Week', 'Jour']).agg(
    rows=('Operacion', 'size'), operations=('Operacion', 'nunique'))
display(coverage)
expected = config.groupby('Linea').size().rename('expected_operations')
line_coverage = joined.groupby(['Linea', 'Year', 'Week', 'Jour']).size().rename('observed_operations').reset_index()
line_coverage = line_coverage.join(expected, on='Linea')
display(line_coverage.loc[line_coverage.observed_operations.ne(line_coverage.expected_operations)])

display(data[VALUES].describe().round(2))


,Count
Missing values (all columns),0
Non-numeric values in numeric columns,0
Non-finite numeric values,0
Negative time/counter values,0
Duplicate daily keys,0
Duplicate configuration keys,0
Missing configuration keys/order,0
Non-integer counters,0


,Linea,Operacion
Unmatched data operations,,


,Linea,Operacion
Configured operations with no observations,,


rows  operations
Year Week Jour                  
2026 40   1       91          91
          2       91          91
          3       91          91
          4       91          91
          5       91          91
     41   1       91          91
          2       91          91
          3       91          91
          4       91          91
          5       91          91
     42   1       91          91
          2       91          91
          3       91          91
          4       91          91
          5       91          91
     43   1       91          91
          2       91          91
          3       91          91
          4       91          91
          5       91          91

,Linea,Year,Week,Jour,observed_operations,expected_operations


,TBF,TAP,DES,SAT,NAP,NPR
count,1820.00,1820.00,1820.00,1820.00,1820.00,1820.00
mean,14.18,5.34,1.23,0.74,245.85,1127.17
std,8.00,8.91,1.57,1.10,2055.11,694.66
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,13.02,0.15,0.00,0.00,0.00,0.00
50%,18.23,0.41,0.90,0.38,17.00,1508.00
75%,19.44,3.13,1.70,1.00,57.00,1542.00
max,22.78,22.01,13.32,8.72,23022.00,3314.00


## 2. Flag suspicious records without deleting them

Zero production is not automatically bad data: a machine may be inactive, stopped,
or use a counter that needs clarification. Production with zero operating time is
inconsistent for cycle-time analysis. A 22-hour total is a reference to investigate,
not a value to force onto every row. Missing intervals and counter resets need client context.

In [4]:
joined['TAI'] = joined['DES'] + joined['SAT']
joined['TR'] = joined[TIMES].sum(axis=1)
joined['production_without_runtime'] = joined['NPR'].gt(0) & joined['TBF'].eq(0)
joined['runtime_without_production'] = joined['TBF'].gt(0) & joined['NPR'].eq(0)
joined['zero_production'] = joined['NPR'].eq(0)
joined['zero_recorded_time'] = joined['TR'].eq(0)
joined['time_outside_reference'] = joined['TR'].sub(EXPECTED_HOURS).abs().gt(TIME_TOLERANCE_HOURS)
# These are investigation signals, not reasons to rewrite counters automatically.
joined['stops_without_own_time'] = joined['NAP'].gt(0) & joined['TAP'].eq(0)
joined['own_time_without_stops'] = joined['TAP'].gt(0) & joined['NAP'].eq(0)
FLAGS = ['production_without_runtime', 'runtime_without_production', 'zero_production',
         'zero_recorded_time', 'time_outside_reference', 'stops_without_own_time', 'own_time_without_stops']
display(joined[FLAGS].sum().rename('Flagged daily records').to_frame())
display(joined.loc[joined.production_without_runtime, DAY_KEYS + VALUES])
display(joined.nlargest(10, 'NAP')[DAY_KEYS + VALUES])
print('Largest stop counters require checking their meaning, reset behavior and units.')
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(joined['TR'], bins=35, color=COLORS['DES'], edgecolor='white')
ax.axvline(EXPECTED_HOURS, color=COLORS['TAP'], linestyle='--', label='22-hour reference')
ax.set(xlabel='Recorded TBF + TAP + DES + SAT (hours / operation / day)', ylabel='Daily records',
       title='Recorded time: inspect gaps and totals above the reference')
ax.legend()
plt.tight_layout()
plt.show()

,Flagged daily records
production_without_runtime,4
runtime_without_production,65
zero_production,463
zero_recorded_time,16
time_outside_reference,1035
stops_without_own_time,16
own_time_without_stops,364


,Linea,Year,Week,Jour,Operacion,TBF,TAP,DES,SAT,NAP,NPR
1185,VLL_MONTAJE_L5,2026,42,4,Ma_BM2_2,0.0,22.00,0.0,0.0,12,1509
1549,VLL_MONTAJE_L5,2026,43,3,Ma_BM2_2,0.0,22.00,0.0,0.0,12,1541
1640,VLL_MONTAJE_L5,2026,43,4,Ma_BM2_2,0.0,22.01,0.0,0.0,12,1513
1731,VLL_MONTAJE_L5,2026,43,5,Ma_BM2_2,0.0,22.00,0.0,0.0,12,1531


,Linea,Year,Week,Jour,Operacion,TBF,TAP,DES,SAT,NAP,NPR
345,VLL_MONTAJE_L5,2026,40,4,ST2235,3.30,14.92,2.81,2.30,23022,2296
1437,VLL_MONTAJE_L5,2026,43,1,ST2235,3.64,15.13,2.19,2.30,22877,2323
1255,VLL_MONTAJE_L5,2026,42,4,ST2235,3.76,15.01,2.15,2.16,22415,2430
436,VLL_MONTAJE_L5,2026,40,5,ST2235,3.60,14.91,2.58,2.19,22411,2347
1073,VLL_MONTAJE_L5,2026,42,2,ST2235,3.87,15.24,2.18,2.02,22281,2468
1801,VLL_MONTAJE_L5,2026,43,5,ST2235,3.84,15.02,2.17,2.24,22199,2402
800,VLL_MONTAJE_L5,2026,41,4,ST2235,3.85,14.59,2.31,2.46,22185,2455
709,VLL_MONTAJE_L5,2026,41,3,ST2235,3.84,14.65,2.36,2.36,22158,2422
1710,VLL_MONTAJE_L5,2026,43,4,ST2235,3.74,14.78,2.13,2.48,21950,2329
1164,VLL_MONTAJE_L5,2026,42,3,ST2235,3.83,14.70,2.29,2.39,21854,2425


Largest stop counters require checking their meaning, reset behavior and units.


C:\Users\User\AppData\Local\Temp\ipykernel_23904\4134791747.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Calculate indicators after summing the selected period

- **Availability (%)** = TBF / (TBF + TAP) x 100. Induced stops do not enter this denominator.
- **Induced stop time (TAI)** = DES + SAT; **recorded required time (TR)** = TBF + TAP + TAI.
- **Own-stop rate** = NAP / NPR x `STOP_RATE_BASE`. This notebook uses **per 100 pieces**
  per the latest instructions. The slides call Fpm the rate **per 1,000 pieces**;
  change the parameter to 1000 if that definition is confirmed. Never compare differently scaled rates.
- **Cycle time (cmin / piece)** = TBF in hours x 6000 / NPR.
- **Mean own-stop duration (seconds)** = TAP in hours x 3600 / NAP.

A zero denominator gives an undefined value, not zero. Keep zero-production days
in accumulated times so availability is not inflated by selecting only productive days.
Do not sum machine piece counters to report line output: serial and parallel operations
may count the same physical pieces or different product flows.

In [5]:
def add_metrics(frame):
    result = frame.copy()
    result['TAI'] = result['DES'] + result['SAT']
    result['TR'] = result[TIMES].sum(axis=1)
    result['availability_pct'] = 100 * result['TBF'].div((result['TBF'] + result['TAP']).where(lambda s: s.gt(0)))
    result['own_stop_rate'] = STOP_RATE_BASE * result['NAP'].div(result['NPR'].where(lambda s: s.gt(0)))
    result['cycle_cmin'] = 6000 * result['TBF'].div(result['NPR'].where(lambda s: s.gt(0)))
    # A period with production but no runtime on any constituent day has suspect cycle data.
    result.loc[result['production_without_runtime'].gt(0) | result['TBF'].le(0), 'cycle_cmin'] = np.nan
    result['mean_own_stop_seconds'] = 3600 * result['TAP'].div(result['NAP'].where(lambda s: s.gt(0)))
    result['induced_pct'] = 100 * result['TAI'].div(result['TR'].where(lambda s: s.gt(0)))
    result['availability_gap_pp'] = (AVAILABILITY_TARGET - result['availability_pct']).clip(lower=0)
    return result

OP_KEYS = ['Linea', 'Operacion', 'Orden_OP']
operations = add_metrics(joined.groupby(OP_KEYS, as_index=False)[VALUES + FLAGS].sum())
weekly = add_metrics(joined.groupby(OP_KEYS + ['Year', 'Week'], as_index=False)[VALUES + FLAGS].sum())
operations = operations.sort_values(['Linea', 'Orden_OP']).reset_index(drop=True)
METRICS = ['Operacion', 'Orden_OP', 'TBF', 'TAP', 'TAI', 'NPR', 'NAP',
           'availability_pct', 'cycle_cmin', 'own_stop_rate', 'mean_own_stop_seconds']
display(operations[METRICS].round(2))
print(f'Stop-rate column is measured in: {RATE_LABEL}')
print('Inactive / no-production operations:')
display(operations.loc[operations.NPR.eq(0), ['Operacion', 'TBF', 'TAP', 'TAI', 'NPR']])
# Sanity checks target mathematical properties rather than implementation details.
assert operations['availability_pct'].dropna().between(0, 100).all()
assert operations['induced_pct'].dropna().between(0, 100).all()
assert not np.isinf(operations.select_dtypes('number').to_numpy()).any()
assert np.isclose(operations['TAP'].sum(), joined['TAP'].sum())

,Operacion,Orden_OP,TBF,TAP,TAI,NPR,NAP,availability_pct,cycle_cmin,own_stop_rate,mean_own_stop_seconds
0,ST_0015,1,273.85,106.21,39.34,24469,747,72.05,67.15,3.05,511.86
1,ST_0020,2,267.15,89.47,83.01,24159,57,74.91,66.35,0.24,5650.74
2,ST_0030,3,0.00,88.00,0.00,0,0,0.00,NaN,NaN,NaN
3,ST_0040,4,290.16,91.01,58.63,24577,151,76.12,70.84,0.61,2169.77
4,ST-0050,5,272.64,92.10,65.03,24794,348,74.75,65.98,1.40,952.76
...,...,...,...,...,...,...,...,...,...,...,...
86,ManBM2_1,87,123.41,316.37,0.00,0,140,28.06,NaN,NaN,8135.23
87,Ma_BM2_2,88,78.07,245.60,116.16,23862,5099,24.12,NaN,21.37,173.40
88,Ma_VES_0,89,351.77,88.05,0.00,0,8,79.98,NaN,NaN,39622.50
89,Ma_VESRE,90,351.77,88.10,0.00,0,3,79.97,NaN,NaN,105720.00


Stop-rate column is measured in: Own stops / 100 pieces
Inactive / no-production operations:


,Operacion,TBF,TAP,TAI,NPR
2,ST_0030,0.00,88.00,0.0,0
48,PM2130,0.00,439.87,0.0,0
85,Ma_BM1_2,351.75,88.11,0.0,0
86,ManBM2_1,123.41,316.37,0.0,0
88,Ma_VES_0,351.77,88.05,0.0,0
89,Ma_VESRE,351.77,88.10,0.0,0


## 4. Action list: own-stop losses first

Among operations with production and operating time, prioritize those below the
90% availability target by their availability deficit, breaking ties by own-stop hours.
This is a transparent investigation order, not an estimated causal impact on line output.
Operations with inconsistent production/runtime records are routed to data review.
Show cycle breaches separately so a fast but unreliable machine is not confused with a slow one.

In [6]:
weekly['availability_breach'] = weekly['NPR'].gt(0) & weekly['availability_pct'].lt(AVAILABILITY_TARGET)
recurrence = weekly.groupby(JOIN_KEYS).agg(
    observed_weeks=('Week', 'size'),
    weeks_with_production=('NPR', lambda s: int(s.gt(0).sum())),
    weeks_below_availability=('availability_breach', 'sum'))
operations = operations.merge(recurrence, on=JOIN_KEYS, how='left', validate='one_to_one')
active = operations.loc[operations['NPR'].gt(0) & operations['TBF'].gt(0)
                        & operations['production_without_runtime'].eq(0)].copy()
priorities = active.loc[active['availability_pct'].lt(AVAILABILITY_TARGET)].sort_values(
    ['availability_gap_pp', 'TAP'], ascending=False).copy()

def action_reason(row):
    reason = (f"Availability {row.availability_pct:.1f}% vs 90%; {row.TAP:.1f} h own stops; "
              f"below target in {int(row.weeks_below_availability)}/{int(row.weeks_with_production)} productive weeks.")
    if pd.notna(row.mean_own_stop_seconds) and row.mean_own_stop_seconds < MICROSTOP_MEAN_SECONDS:
        reason += ' Short mean own-stop duration: inspect event logs; microstops unconfirmed.'
    else:
        reason += ' Review own-stop causes and repair records with maintenance.'
    if row.time_outside_reference > 0:
        reason += ' Verify recorded-time coverage before deciding.'
    return reason

priorities['reason_and_next_step'] = priorities.apply(action_reason, axis=1)
ACTION_COLUMNS = ['Operacion', 'Orden_OP', 'availability_pct', 'TAP', 'induced_pct',
                  'own_stop_rate', 'mean_own_stop_seconds', 'weeks_below_availability',
                  'weeks_with_production', 'reason_and_next_step']
display(priorities[ACTION_COLUMNS].head(10).round(2))
print('Slow-cycle investigation (80 cmin = 48 seconds per piece):')
slow_cycles = active.loc[active.cycle_cmin.gt(CYCLE_TARGET_CMIN)].sort_values('cycle_cmin', ascending=False)
display(slow_cycles[['Operacion', 'cycle_cmin', 'availability_pct', 'NPR']].round(2))
print('Data review before operational ranking:')
display(operations.loc[operations.production_without_runtime.gt(0),
                       ['Operacion', 'NPR', 'TBF', 'production_without_runtime']])

,Operacion,Orden_OP,availability_pct,TAP,induced_pct,own_stop_rate,mean_own_stop_seconds,weeks_below_availability,weeks_with_production,reason_and_next_step
60,ST2235,61,15.60,324.95,16.06,924.62,3.32,4,4,Availability 15.6% vs 90%; 324.9 h own stops; ...
90,MANOP749,91,54.08,165.28,17.32,57.13,71.04,4,4,Availability 54.1% vs 90%; 165.3 h own stops; ...
74,ST-2390,75,59.80,156.26,9.54,39.33,58.41,4,4,Availability 59.8% vs 90%; 156.3 h own stops; ...
62,PM2250,63,66.31,134.63,5.93,1.54,1278.81,4,4,Availability 66.3% vs 90%; 134.6 h own stops; ...
7,ST_0100,8,69.24,119.07,9.81,6.53,265.75,3,4,Availability 69.2% vs 90%; 119.1 h own stops; ...
33,ManutPul,34,70.94,90.37,29.30,27.32,48.35,3,4,Availability 70.9% vs 90%; 90.4 h own stops; b...
39,ST-2040,40,71.16,109.32,12.05,7.55,214.59,3,4,Availability 71.2% vs 90%; 109.3 h own stops; ...
19,ST-0260,20,71.20,98.90,9.57,4.05,356.04,3,4,Availability 71.2% vs 90%; 98.9 h own stops; b...
18,ST0259,19,71.76,92.78,25.32,1.42,951.59,3,4,Availability 71.8% vs 90%; 92.8 h own stops; b...
0,ST_0015,1,72.05,106.21,9.38,3.05,511.86,3,4,Availability 72.1% vs 90%; 106.2 h own stops; ...


Slow-cycle investigation (80 cmin = 48 seconds per piece):


,Operacion,cycle_cmin,availability_pct,NPR
24,ST-0295,44404.39,77.26,41
71,ST2376,178.89,77.12,11732
73,ST-2375,152.58,76.88,12454
78,ST-0830,85.27,78.97,24462
5,PM_0090,81.80,77.45,24854
81,ST_3100,81.21,77.54,24304
46,PM2110,80.78,77.71,23885


Data review before operational ranking:


,Operacion,NPR,TBF,production_without_runtime
87,Ma_BM2_2,23862,78.07,4


In [7]:
# Show the decision, then its supporting time breakdown.
top = priorities.head(10).sort_values('availability_pct', ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(top['Operacion'], top['availability_pct'], color=COLORS['TAP'])
axes[0].axvline(AVAILABILITY_TARGET, linestyle='--', color='#333333', label='90% target')
axes[0].set(xlim=(0, 100), xlabel='Availability (%)', title='Priority operations: own-stop availability')
axes[0].legend()
time_breakdown = top.set_index('Operacion')[['TAP', 'DES', 'SAT']]
time_breakdown.plot.barh(stacked=True, ax=axes[1], color=[COLORS[c] for c in time_breakdown.columns])
axes[1].set(xlabel='Accumulated machine-hours (four weeks)', ylabel='',
            title='Own stops versus external waiting on the same operations')
plt.tight_layout()
plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_23904\1733729823.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Recurrence: distinguish persistent problems from isolated weeks

Use the same availability calculation after summing each week. Repeated breaches
support a recurring investigation priority. Four weeks do not establish seasonality
or prove that a future fault will recur. After an intervention, compare these same
indicators over comparable production periods to check whether it helped.

In [8]:
selected = priorities.head(10)['Operacion'].tolist()
heatmap = weekly.loc[weekly['Operacion'].isin(selected)].pivot(
    index='Operacion', columns=['Year', 'Week'], values='availability_pct').reindex(selected)
if not heatmap.empty:
    fig, ax = plt.subplots(figsize=(9, max(3, len(heatmap) * 0.45)))
    image = ax.imshow(heatmap.to_numpy(dtype=float), cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(len(heatmap.columns)), [f'{year}-W{week}' for year, week in heatmap.columns])
    ax.set_yticks(range(len(heatmap)), heatmap.index)
    for i in range(len(heatmap)):
        for j in range(len(heatmap.columns)):
            value = heatmap.iloc[i, j]
            ax.text(j, i, f'{value:.1f}%' if pd.notna(value) else 'N/A', ha='center', va='center', fontsize=9)
    ax.set_title('Do the priority machines miss the 90% target repeatedly?')
    fig.colorbar(image, ax=ax, label='Availability (%)')
    plt.tight_layout()
    plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_23904\3245281781.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Microstop screening: frequency versus average duration

A daily counter cannot detect individual microstops. A high stop rate combined with
a low **mean** duration suggests where to inspect event-level logs, sensors and resets.
The 60-second threshold below is an editable exploration heuristic, not a client rule.
A mean can hide a mixture of long breakdowns and short interruptions. Also check
whether NAP really counts own stops: unusually large counters can dominate this signal.

In [9]:
stop_candidates = active.loc[active['NAP'].gt(0) & active['TAP'].gt(0)].copy()
frequent_threshold = stop_candidates['own_stop_rate'].quantile(0.75)
stop_candidates['microstop_signal'] = (
    stop_candidates['own_stop_rate'].ge(frequent_threshold)
    & stop_candidates['mean_own_stop_seconds'].lt(MICROSTOP_MEAN_SECONDS))
print(f'Frequent-stop screen: upper quartile, >= {frequent_threshold:.2f} {RATE_LABEL.lower()}')
display(stop_candidates.loc[stop_candidates.microstop_signal,
    ['Operacion', 'NAP', 'TAP', 'own_stop_rate', 'mean_own_stop_seconds']]
    .sort_values('own_stop_rate', ascending=False).round(2))
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(stop_candidates['own_stop_rate'], stop_candidates['mean_own_stop_seconds'],
           c=np.where(stop_candidates.microstop_signal, COLORS['TAP'], COLORS['DES']), alpha=0.75)
ax.axhline(MICROSTOP_MEAN_SECONDS, linestyle='--', color='#555555', label='Mean-duration screen: 60 s')
ax.axvline(frequent_threshold, linestyle=':', color='#555555', label='Stop-rate upper quartile')
for row in stop_candidates.nlargest(5, 'own_stop_rate').itertuples():
    ax.annotate(row.Operacion, (row.own_stop_rate, row.mean_own_stop_seconds), xytext=(4, 5), textcoords='offset points')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set(xlabel=RATE_LABEL + ' (log scale)', ylabel='Mean own-stop duration (seconds, log scale)',
       title='Frequent, short interruptions: investigation signals, not confirmed microstops')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Frequent-stop screen: upper quartile, >= 5.29 own stops / 100 pieces


,Operacion,NAP,TAP,own_stop_rate,mean_own_stop_seconds
60,ST2235,352355,324.95,924.62,3.32
74,ST-2390,9630,156.26,39.33,58.41
33,ManutPul,6729,90.37,27.32,48.35


C:\Users\User\AppData\Local\Temp\ipykernel_23904\2881653134.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. External waiting and the line configuration

DES means an operation lacks incoming supply; SAT means it cannot pass output onward.
Use these patterns to choose where to investigate, together with own stops and line
knowledge. Neither adjacent order numbers nor low TAI prove which operation caused a
stop, especially with parallel branches. Obtain branch connectivity and event timestamps
before attempting propagation attribution.

In [10]:
ordered = operations.sort_values(['Linea', 'Orden_OP']).set_index('Operacion')
percentages = ordered[TIMES].div(ordered['TR'].where(ordered['TR'].gt(0)), axis=0) * 100
ax = percentages.plot.bar(stacked=True, figsize=(19, 6), width=0.85,
                           color=[COLORS[c] for c in TIMES])
ax.set(xlabel='Operations in configuration order (parallel paths are not shown)',
       ylabel='Share of recorded time (%)', ylim=(0, 100),
       title='Line overview: operating time, own stops and induced waiting')
ax.tick_params(axis='x', labelsize=7)
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.12))
plt.tight_layout()
plt.show()
print('High induced waiting: investigate supply/blocking; not necessarily the source of lost output.')
display(active.nlargest(10, 'TAI')[['Operacion', 'Orden_OP', 'DES', 'SAT', 'TAI', 'availability_pct']].round(2))
print('Low induced waiting: challenge clue, to cross-check with own stops and cycle time.')
display(active.nsmallest(10, 'induced_pct')[['Operacion', 'Orden_OP', 'induced_pct', 'TAP', 'availability_pct', 'cycle_cmin']].round(2))

High induced waiting: investigate supply/blocking; not necessarily the source of lost output.


C:\Users\User\AppData\Local\Temp\ipykernel_23904\2512101554.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Operacion,Orden_OP,DES,SAT,TAI,availability_pct
33,ManutPul,34,126.59,2.27,128.86,70.94
18,ST0259,19,3.45,107.91,111.36,71.76
83,ST_3300,84,108.42,0.00,108.42,73.24
21,ST-0270,22,82.46,19.42,101.88,72.28
1,ST_0020,2,27.93,55.08,83.01,74.91
16,ST-0250,17,7.52,67.97,75.49,73.91
90,MANOP749,91,41.73,33.68,75.41,54.08
14,ST-0230,15,14.02,61.38,75.40,74.78
60,ST2235,61,37.21,36.44,73.65,15.60
20,ST_0265,21,66.85,5.45,72.30,74.87


Low induced waiting: challenge clue, to cross-check with own stops and cycle time.


,Operacion,Orden_OP,induced_pct,TAP,availability_pct,cycle_cmin
36,PM-2020,37,1.40,99.74,76.08,77.68
17,ST0261,18,1.91,95.22,76.77,75.23
78,ST-0830,79,3.00,92.57,78.97,85.27
71,ST2376,72,3.21,103.79,77.12,178.89
45,ST-2100,46,4.16,105.49,74.96,78.07
46,PM2110,47,4.69,92.22,77.71,80.78
32,ST-1000,33,4.77,90.37,76.16,72.03
81,ST_3100,82,4.79,95.26,77.54,81.21
59,PM2230,60,5.29,93.25,75.40,69.83
70,ST-2360,71,5.33,96.27,76.87,78.26


## 8. A concise maintenance briefing

The output should say **where to go, why, what to check, and how to monitor improvement**.
The statements below are generated from the data rather than hard-coded machine names.
No claim is made that all own-stop hours are recoverable production hours for the line.

In [11]:
print(f'Coverage: {joined.Operacion.nunique()} operations, {joined.Week.nunique()} weeks, '
      f'{len(coverage)} observed production days.')
print(f'{len(priorities)} active, consistent operations are below {AVAILABILITY_TARGET:.0f}% availability.')
if len(priorities) == len(active) and len(active) > 0:
    print('All eligible active operations miss the target: validate the baseline and time classification before treating every operation as a separate maintenance fault.')
for row in priorities.head(3).itertuples():
    print(f'\nACTION: {row.Operacion} (configuration order {row.Orden_OP})')
    print(row.reason_and_next_step)
    print('Follow-up: compare weekly availability, own-stop hours and normalized stop rate after intervention.')
print(f'\n{len(slow_cycles)} active operations exceed {CYCLE_TARGET_CMIN:.0f} cmin/piece under the hours assumption.')
print(f'{int(stop_candidates.microstop_signal.sum())} operations match the exploratory frequent/short-stop screen.')
print('Confirm time units, per-100 vs per-1000 rate, counter semantics, inactive operations and parallel routes.')

Coverage: 91 operations, 4 weeks, 20 observed production days.
84 active, consistent operations are below 90% availability.
All eligible active operations miss the target: validate the baseline and time classification before treating every operation as a separate maintenance fault.

ACTION: ST2235 (configuration order 61)
Availability 15.6% vs 90%; 324.9 h own stops; below target in 4/4 productive weeks. Short mean own-stop duration: inspect event logs; microstops unconfirmed. Verify recorded-time coverage before deciding.
Follow-up: compare weekly availability, own-stop hours and normalized stop rate after intervention.

ACTION: MANOP749 (configuration order 91)
Availability 54.1% vs 90%; 165.3 h own stops; below target in 4/4 productive weeks. Review own-stop causes and repair records with maintenance. Verify recorded-time coverage before deciding.
Follow-up: compare weekly availability, own-stop hours and normalized stop rate after intervention.

ACTION: ST-2390 (configuration order

## Delivery plan and limits

**Minimal repository:** keep this exploration notebook; use `cleaning.py` for loading,
validation and the configuration join; use one future `analysis.py` for indicators,
action tables and figures. No interface is required. Leave existing app code untouched
until the team decides whether to remove it. Do not add machine learning for this scope.

**Eight-minute presentation:** problem and objective (1 min), data and assumptions
(1 min), three machines needing attention and their evidence (3 min), recurring issues
and microstop screening (2 min), next actions and limitations (1 min). Reserve detail
on validation and formulas for questions. Presentations are private.

**Original but feasible idea:** a maintenance action card for each priority machine,
with its own-stop losses, recurrence across weeks, short-stop signal, a suggested
inspection and before/after monitoring. Describe sensor checks as hypotheses, not
fault diagnoses. Preventing recurrence requires interventions and follow-up data;
this dataset alone does not support predicting failures.

**Before delivery:** confirm hours for accumulated times; clarify the stop-rate base
(100 in latest notes, 1000 in slides); validate high NAP counters and zero-production
operations; confirm targets and parallel connectivity. Preserve raw inputs, and do
not silently remove unusual records or promise exact root-cause attribution.